# Intelligent Resource Governor (IRG) - Analytics & Validation Engine

This notebook acts as the academic and statistical sandbox for validating the Predictive Edge Middleware. It processes historical resource telemetry datasets, calculates system overload risks, trains the governing brain (Random Forest Classifier), and evaluates model features.

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'ggplot')
print("Libraries imported successfully.")

## 2. Load Telemetry Data
We load the generated `server_logs.csv` dataset capturing CPU utilization, Memory allocations, and response metrics.

In [ ]:
df = pd.read_csv('server_logs.csv')
df.head(10)

## 3. Exploratory Data Analysis
Plot the telemetry variables (CPU, Memory, Latency) across the 30-minute timeline, displaying where resource exhaustion points occur.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(df['timestamp'], df['cpu_utilization'], color='tab:red', label='CPU %')
axes[0].axhline(80, color='red', linestyle='--', label='80% Stress Threshold')
axes[0].set_ylabel('CPU Utilization (%)')
axes[0].legend(loc='upper right')
axes[0].set_title('Infrastructure Load Telemetry')

axes[1].plot(df['timestamp'], df['ram_utilization'], color='tab:blue', label='RAM %')
axes[1].axhline(80, color='blue', linestyle='--')
axes[1].set_ylabel('RAM Utilization (%)')
axes[1].legend(loc='upper right')

axes[2].plot(df['timestamp'], df['latency_ms'], color='tab:orange', label='Latency (ms)')
axes[2].set_ylabel('Response Latency (ms)')
axes[2].set_xlabel('Elapsed Time (Seconds)')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 4. Train Predictive Model
Split features and labels, train the Random Forest Classifier, and export the binary model file `governor_model.pkl`.

In [ ]:
features = ['cpu_utilization', 'ram_utilization', 'request_rate', 'context_switches', 'worker_threads']
X = df[features]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, 'governor_model.pkl')
print("Model trained and serialized as 'governor_model.pkl'.")

## 5. Model Evaluation and Insights

In [ ]:
y_pred = model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
disp = ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=['Stable', 'Governed'], cmap=plt.cm.Blues)
plt.title('Predictive Governor Confusion Matrix')
plt.grid(False)
plt.show()

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 5))
plt.title('Feature Importances for Infrastructure Failure Risk')
plt.barh(range(len(indices)), importances[indices], color='tab:blue', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()